In [ ]:
# @title 🧠 Dependencies { display-mode: "form" }
# Box 1: Clean install for Colab
!pip uninstall -y google-adk bigframes opentelemetry-exporter-gcp-logging wandb
!pip install -q "crewai[tools]" "langchain-google-genai" "requests==2.32.4" "langchain" "langchain-openrouter"

print("✅ Clean CrewAI environment installed.")

Found existing installation: google-adk 1.29.0
Uninstalling google-adk-1.29.0:
  Successfully uninstalled google-adk-1.29.0
Found existing installation: bigframes 2.41.0
Uninstalling bigframes-2.41.0:
  Successfully uninstalled bigframes-2.41.0
Found existing installation: opentelemetry-exporter-gcp-logging 1.11.0a0
Uninstalling opentelemetry-exporter-gcp-logging-1.11.0a0:
  Successfully uninstalled opentelemetry-exporter-gcp-logging-1.11.0a0
Found existing installation: wandb 0.27.1
Uninstalling wandb-0.27.1:
  Successfully uninstalled wandb-0.27.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# @title 🧠 Executive Intelligence Control Center { display-mode: "form" }
import os
import html
import warnings
import requests

warnings.filterwarnings("ignore")

# ==========================================================
# 1. LOAD SECRETS SAFELY FROM COLAB
# ==========================================================
from google.colab import userdata

# Load GOOGLE_API_KEY from Colab secrets
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
# Load SERPER_API_KEY from Colab secrets
SERPER_API_KEY = userdata.get("SERPER_API_KEY")
# It's also good practice to load these from secrets if they are sensitive
TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID = userdata.get("TELEGRAM_CHAT_ID")


# Optional only if you truly need OpenRouter later
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY") or ""

# ==========================================================
# 2. ENVIRONMENT VARIABLES
# ==========================================================
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["SERPER_API_KEY"] = SERPER_API_KEY
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["CREWAI_DISABLE_TRACING"] = "true"

if OPENROUTER_API_KEY:
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# ==========================================================
# 3. SAFE TELEGRAM MESSAGE FUNCTION
# ==========================================================

# Helper function to escape MarkdownV2 special characters for Telegram
def escape_markdown_v2_final(text):
    # Escape backslashes first to prevent double escaping
    escaped_text = text.replace('\\', '\\\\') # Escape existing backslashes
    # Escape other MarkdownV2 special characters
    for char in r'_*[]()~`>#+-=|{}.!':
        escaped_text = escaped_text.replace(char, '\\' + char)
    return escaped_text

def send_to_phone(text):
    if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
        print("❌ Telegram sender not configured. Please set TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID in Colab secrets.")
        return None

    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"

    # Removed html.escape to allow direct HTML formatting in the message
    # Escape special characters in the briefing content before adding new formatting
    safe_text = escape_markdown_v2_final(str(text)[:4000])

    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": safe_text,
        "parse_mode": "MarkdownV2", # Changed to MarkdownV2
        "disable_web_page_preview": True
    }

    try:
        response = requests.post(url, json=payload, timeout=30)
        print("Telegram status:", response.status_code)
        print("Telegram response:", response.text)
        return response
    except Exception as e:
        print(f"❌ Telegram Network Error: {e}")
        return None

print("✅ Configuration loaded and Telegram sender ready.")

✅ Configuration loaded and Telegram sender ready.


In [ ]:
# @title 🧠 AGENTS RUNTIME { display-mode: "form" }
import warnings
warnings.filterwarnings("ignore")

from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool

from langchain_openrouter import ChatOpenRouter
import os

from datetime import datetime, timedelta

# Get current date
current_date = datetime.now()
# Get date 2 days ago
two_days_ago = current_date - timedelta(days=2)
# Format dates for the prompt
current_date_str = current_date.strftime("%Y-%m-%d")
two_days_ago_str = two_days_ago.strftime("%Y-%m-%d")

# 1. Notify your phone that the workflow started
send_to_phone("🚀 Intelligence Matrix Initialized: starting research pipeline.")


# 2. Create a real search tool
search_tool = SerperDevTool()

# Use OpenRouter instead of Gemini
# You will need to choose a model that OpenRouter supports
# Examples: "google/gemini-pro-1.5", "openai/gpt-4o", "mistralai/mistral-large"
openrouter_llm = ChatOpenRouter(
    model_name="openrouter/owl-alpha", # <--- IMPORTANT: Replace with an actual model name
    openrouter_api_key=os.environ["OPENROUTER_API_KEY"],
    verbose=True
)

# 3. Define agents
researcher = Agent(
    role="Current Affairs Analyst", # Renamed agent role
    goal="Search the live web for the most recent and critical updates on specified topics.", # Updated goal
    backstory="You are an expert in sifting through vast amounts of real-time web information to provide up-to-the-minute factual research across macroeconomics, geopolitics, and strategic thinking. You focus on the very latest developments.", # Updated backstory
    tools=[search_tool],
    llm=openrouter_llm, # Use the OpenRouter LLM
    verbose=True
)

writer = Agent(
    role="Executive Communications Director",
    goal="Turn research into a concise and structured executive brief.",
    backstory="You write for busy leaders and focus on signal, structure, and clarity.",
    llm=openrouter_llm, # Use the OpenRouter LLM
    verbose=True
)

critic = Agent(
    role="Senior Editorial Director",
    goal="Refine the draft for balance, density, and clarity.",
    backstory="You remove fluff, improve structure, and ensure the final brief is practical and sharp.",
    llm=openrouter_llm, # Use the OpenRouter LLM
    verbose=True
)

# 4. Define tasks
task1 = Task(
    description="""
Investigate the following three topics using live web search:

{topic}

For each section:
- identify the most important recent developments,
- keep the findings factual and specific,
- STRICTLY ADHERE to the date ranges specified for each topic in the '{topic}' input. Discard any information that falls outside these specified date windows.
- include source names or links in plain text where useful.
""",
    expected_output="A factual research summary covering all three domains with concrete findings, strictly adhering to the specified date ranges for each topic.",
    agent=researcher
)

task2 = Task(
    description="""
Take the research and turn it into a structured executive draft.

Requirements:
- separate the three topics clearly,
- keep the writing concise and professional,
- focus on implications, not just facts.
""",
    expected_output="A structured executive draft with three clearly separated sections.",
    agent=writer
)

task3 = Task(
    description="""
Review the writer's draft and improve it.

Requirements:
1. Balance all three sections: Global Macro, Geopolitics, and Smart Thinking.
2. Remove filler, sensationalism, and repetition.
3. Return plain text only.
4. Keep the full brief under 4000 characters.
5. Use this exact structure:

GLOBAL MACRO
- Use relevant emojis for bullet points (with date where appropriate for news)

GEOPOLITICS
- Use relevant emojis for bullet points (with date where appropriate for news)

SMART THINKING
- Use relevant emojis for bullet points

SOURCES
- List of relevant links in Markdown format (e.g., [Source Name](URL)).
""",
    expected_output="A final plain-text executive brief under 400 words with three sections, emoji-based bullet points including dates for news in Global Macro and Geopolitics, and a 'SOURCES' section with relevant links in Markdown format.",
    agent=critic
)

# 5. Build crew
crew = Crew(
    agents=[researcher, writer, critic],
    tasks=[task1, task2, task3],
    process=Process.sequential,
    verbose=True
)



# 6. Run synchronously first for reliability in Colab
try:
    print("Running multi-topic agent pipeline...")

    result = crew.kickoff(inputs={
        "topic": f"""
1. GLOBAL MACRO TRENDS: Central bank interest rate adjustments, global inflation dynamics, and sovereign debt shifts. Focus on developments between {two_days_ago_str} and {current_date_str}.
2. GLOBAL GEOPOLITICS: Maritime trade route friction, energy supply chain security, and strategic tech/industrial alliances in East Asia or Europe. Focus on developments between {two_days_ago_str} and {current_date_str}.
3. SMART THINKING & MENTAL MODELS: One Influential High-level cognitive framework or strategic decision-making model, or modern systems-thinking insights from top analytical minds. Focus on developments from the past 1 month from {current_date_str}.
"""
    })

    final_briefing = getattr(result, "raw", str(result))

    # Escape special characters in the briefing content before adding new formatting
    # final_briefing = escape_markdown_v2_final(final_briefing)

    # Add Markdown formatting and emojis to section headers
    # These replacements will now correctly introduce unescaped MarkdownV2 characters
    final_briefing = final_briefing.replace("GLOBAL MACRO", "**🌍 GLOBAL MACRO**")
    final_briefing = final_briefing.replace("GEOPOLITICS", "**🌐 GEOPOLITICS**")
    final_briefing = final_briefing.replace("SMART THINKING", "**🧠 SMART THINKING**")

    message_content = (
        "**DAILY INTELLIGENCE BRIEFING**\n\n"
        + final_briefing
    )

    send_to_phone(message_content)
    print("✅ Success! Briefing sent to Telegram.")

except Exception as e:
    print(f"\n❌ CRITICAL ERROR LOGGED:\n{str(e)}")
    send_to_phone(f"Agent execution error: {str(e)}")

Telegram status: 200
Telegram response: {"ok":true,"result":{"message_id":100,"from":{"id":8780596586,"is_bot":true,"first_name":"CHARLIE","username":"CasalieAgentBot"},"chat":{"id":8866968968,"first_name":"Emanuele","type":"private"},"date":1781172381,"text":"\ud83d\ude80 Intelligence Matrix Initialized: starting research pipeline."}}
Running multi-topic agent pipeline...


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b4f52fd4-ac60-44fb-83ed-88acde91a90a                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Current Affairs Analyst                                                                                 │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Investigate the following three topics using live web search:                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  1. GLOBAL MACRO TRENDS: Central bank interest rate adjustments, global inflation dynamics, and sovereign debt  │
│  shifts. Focus on developments between 2026-06-09 and 2026-06-11.                                               │
│  2. GLOBAL GEOPOLITICS: Maritime trade route friction, energy supply chain security, and strategic              │
│  tech/industrial alliances in East Asia or Europe. Focus on developments between 2026-06-09 and 2026-06-11.     │
│  3. SMART THINKING & MENTAL MODELS: One Influential High-level cognitive framework or strategic                 │
│  decision-making model, or modern systems-thinking insights from top analytical minds. Focus on developments    │
│  from the past 1 month from 2026-06-11.                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│  For each section:                                                                                              │
│  - identify the most important recent developments,                                                             │
│  - keep the findings factual and specific,                                                                      │
│  - STRICTLY ADHERE to the date ranges specified for each topic in the '                                         │
│  1. GLOBAL MACRO TRENDS: Central bank interest rate adjustments, global inflation dynamics, and sovereign debt  │
│  shifts. Focus on developments between 2026-06-09 and 2026-06-11.                                               │
│  2. GLOBAL GEOPOLITICS: Maritime trade route friction, energy supply chain security, and strategic              │
│  tech/industrial alliances in East Asia or Europe. Focus on developments between 2026-06-09 and 2026-06-11.     │
│  3. SMART THINKING & MENTAL MODELS: One Influential High-level cognitive framework or strategic                 │
│  decision-making model, or modern systems-thinking insights from top analytical minds. Focus on developments    │
│  from the past 1 month from 2026-06-11.                                                                         │
│  ' input. Discard any information that falls outside these specified date windows.                              │
│  - include source names or links in plain text where useful.                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Current Affairs Analyst                                                                                 │
│                                                                                                                 │
│  Thought: Thought: I need to investigate three distinct topics with specific date ranges. Let me start by       │
│  searching for each topic simultaneously.                                                                       │
│  For Topic 1 (Global Macro Trends), I need to search for central bank interest rate adjustments, global         │
│  inflation dynamics, and sovereign debt shifts between 2026-06-09 and 2026-06-11.                               │
│  For Topic 2 (Global Geopolitics), I need to search for maritime trade route friction, energy supply chain      │
│  security, and strategic tech/industrial alliances in East Asia or Europe between 2026-06-09 and 2026-06-11.    │
│  For Topic 3 (Smart Thinking & Mental Models), I need to search for influential cognitive frameworks,           │
│  strategic decision-making models, or systems-thinking insights from the past 1 month (from 2026-05-11 to       │
│  2026-06-11).                                                                                                   │
│  Let me start with searches for all three topics.                                                               │
│                                                                                                                 │
│  Using Tool: Search the internet with Serper                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Editorial Director                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  GLOBAL MACRO                                                                                                   │
│  - 🏦 Jun 9: Bank of Canada holds at 2.25% (5th consecutive hold); cites Iran war energy volatility keeping     │
│  inflation near 3%                                                                                              │
│  - 🏦 Jun 2026: Fed holds at 3.50%–3.75%, signals potential cut later in 2026                                   │
│  - 🏦 Jun 2026: Bank of England holds at 3.75% as UK inflation rises to 3.3%; next decision Jun 18              │
│  - 📈 IMF projects global inflation ticking up in 2026 before resuming decline in 2027; J.P. Morgan flags       │
│  destabilized expectations                                                                                      │
│  - 💰 OECD: Global borrowing to hit $29T in 2026 (+17% vs 2024); interest costs to add 2.5pp to debt-to-GDP     │
│  ratios                                                                                                         │
│                                                                                                                 │
│  GEOPOLITICS                                                                                                    │
│  - 🚢 Jun 10: Strait of Hormuz remains blocked since Feb 28; ~1,550 vessels and 22,500 mariners stranded;       │
│  Trump threatens further military action                                                                        │
│  - ⚡ Jun 10: ASEAN energy regulators call for stronger regional cooperation; crisis may catalyze fundamental   │
│  energy policy shifts                                                                                           │
│  - 🔧 Jun 2026: European Commission advances Chips Act 2.0; Chip 4 alliance (US, Japan, SK, Taiwan) anchors     │
│  semiconductor supply chain security                                                                            │
│  - 🤝 Jun 9–10: Aspen Institute Germany hosts German-American Trade & Tech Conference on balancing economic     │
│  openness with strategic tech autonomy                                                                          │
│                                                                                                                 │
│  SMART THINKING                                                                                                 │
│  - 🧠 UCL School of Management introduces "AI Thinking Quadrants" framework (whitepaper: *Cognitive Governance  │
│  in the AI Era*, Apr 2026)                                                                                      │
│  - 🧠 Four cognitive postures map how leaders engage AI in strategic work—from passive delegation to            │
│  co-creative reasoning                                                                                          │
│  - 🧠 Framework addresses core question: How should leaders think strategically when AI shapes the cognitive    │
│  process itself?                                                                                                │
│  - 🧠 Bridges cognitive science, management theory, and AI governance; applicable as a diagnostic tool for      │
│  leadership teams                                                                                               │
│                                                                     

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: bd7ebf05-4a65-4481-8d5b-1eb3f2404339                                                                     │
│  Agent: Senior Editorial Director                                                                               │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b4f52fd4-ac60-44fb-83ed-88acde91a90a                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: GLOBAL MACRO                                                                                     │
│  - 🏦 Jun 9: Bank of Canada holds at 2.25% (5th consecutive hold); cites Iran war energy volatility keeping     │
│  inflation near 3%                                                                                              │
│  - 🏦 Jun 2026: Fed holds at 3.50%–3.75%, signals potential cut later in 2026                                   │
│  - 🏦 Jun 2026: Bank of England holds at 3.75% as UK inflation rises to 3.3%; next decision Jun 18              │
│  - 📈 IMF projects global inflation ticking up in 2026 before resuming decline in 2027; J.P. Morgan flags       │
│  destabilized expectations                                                                                      │
│  - 💰 OECD: Global borrowing to hit $29T in 2026 (+17% vs 2024); interest costs to add 2.5pp to debt-to-GDP     │
│  ratios                                                                                                         │
│                                                                                                                 │
│  GEOPOLITICS                                                                                                    │
│  - 🚢 Jun 10: Strait of Hormuz remains blocked since Feb 28; ~1,550 vessels and 22,500 mariners stranded;       │
│  Trump threatens further military action                                                                        │
│  - ⚡ Jun 10: ASEAN energy regulators call for stronger regional cooperation; crisis may catalyze fundamental   │
│  energy policy shifts                                                                                           │
│  - 🔧 Jun 2026: European Commission advances Chips Act 2.0; Chip 4 alliance (US, Japan, SK, Taiwan) anchors     │
│  semiconductor supply chain security                                                                            │
│  - 🤝 Jun 9–10: Aspen Institute Germany hosts German-American Trade & Tech Conference on balancing economic     │
│  openness with strategic tech autonomy                                                                          │
│                                                                                                                 │
│  SMART THINKING                                                                                                 │
│  - 🧠 UCL School of Management introduces "AI Thinking Quadrants" framework (whitepaper: *Cognitive Governance  │
│  in the AI Era*, Apr 2026)                                                                                      │
│  - 🧠 Four cognitive postures map how leaders engage AI in strategic work—from passive delegation to            │
│  co-creative reasoning                                                                                          │
│  - 🧠 Framework addresses core question: How should leaders think strategically when AI shapes the cognitive    │
│  process itself?                                                                                                │
│  - 🧠 Bridges cognitive science, management theory, and AI governance; applicable as a diagnostic tool for      │
│  leadership teams                                                  

Telegram status: 200
Telegram response: {"ok":true,"result":{"message_id":101,"from":{"id":8780596586,"is_bot":true,"first_name":"CHARLIE","username":"CasalieAgentBot"},"chat":{"id":8866968968,"first_name":"Emanuele","type":"private"},"date":1781172646,"text":"**DAILY INTELLIGENCE BRIEFING**\n\n**\ud83c\udf0d GLOBAL MACRO**\n- \ud83c\udfe6 Jun 9: Bank of Canada holds at 2.25% (5th consecutive hold); cites Iran war energy volatility keeping inflation near 3%\n- \ud83c\udfe6 Jun 2026: Fed holds at 3.50%\u20133.75%, signals potential cut later in 2026\n- \ud83c\udfe6 Jun 2026: Bank of England holds at 3.75% as UK inflation rises to 3.3%; next decision Jun 18\n- \ud83d\udcc8 IMF projects global inflation ticking up in 2026 before resuming decline in 2027; J.P. Morgan flags destabilized expectations\n- \ud83d\udcb0 OECD: Global borrowing to hit $29T in 2026 (+17% vs 2024); interest costs to add 2.5pp to debt-to-GDP ratios\n\n**\ud83c\udf10 GEOPOLITICS**\n- \ud83d\udea2 Jun 10: Strait of Hor